# Sistematización de Iniciativas Populares del proceso constituyente 2023 mediante técnicas de PLN

**Autor:** Dante Aguirre Bravo  
**Curso:** Procesamiento de Lenguaje Natural  
**Programa:** Magíster en Data Science e Inteligencia Artificial  

## Objetivo general

El objetivo de este notebook es sistematizar los textos de las Iniciativas Populares del proceso constituyente 2023 mediante técnicas de Procesamiento de Lenguaje Natural (PLN), combinando métodos exploratorios, supervisados y no supervisados.

El énfasis metodológico del trabajo está puesto en técnicas interpretables: auditoría de la base, descripción de metadatos, frecuencias, bigramas, TF-IDF, diccionario temático constitucional propio, KWIC y visualizaciones. Los modelos de tópicos, embeddings y BERTopic se incluyen como análisis complementarios, porque los textos son breves y la base ya cuenta con clasificaciones institucionales por comisión y capítulo.


## 1. Configuración del entorno

Esta sección prepara el entorno de ejecución, fija una semilla de reproducibilidad y crea las carpetas de salida. La celda de instalación queda comentada porque puede ser útil en Google Colab, pero no se ejecuta automáticamente para evitar modificar entornos locales ya configurados.

**Decisión metodológica:** el notebook debe ejecutarse de principio a fin aun cuando algunas librerías avanzadas no estén instaladas. Por eso, los métodos principales usan librerías estándar del análisis de datos y scikit-learn, mientras que embeddings y BERTopic se controlan con interruptores explícitos.


In [ ]:
# En Google Colab, descomentar si faltan dependencias:
# %pip install -q pandas numpy matplotlib seaborn plotly wordcloud networkx scikit-learn nltk sentence-transformers bertopic umap-learn hdbscan python-docx openpyxl

from pathlib import Path
from collections import Counter, defaultdict
from textwrap import dedent
import itertools
import json
import math
import re
import time
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import networkx as nx
from wordcloud import WordCloud

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances

try:
    import nltk
    from nltk.corpus import stopwords
    try:
        STOPWORDS_NLTK = set(stopwords.words("spanish"))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        STOPWORDS_NLTK = set(stopwords.words("spanish"))
except Exception:
    STOPWORDS_NLTK = set()

try:
    from sentence_transformers import SentenceTransformer
    import umap
    HAS_EMBEDDINGS = True
except Exception:
    HAS_EMBEDDINGS = False

try:
    from bertopic import BERTopic
    from hdbscan import HDBSCAN
    from umap import UMAP
    HAS_BERTOPIC = True
except Exception:
    HAS_BERTOPIC = False

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13

RANDOM_STATE = 42
DATA_PATH = Path("data/raw/IPN_19_07_2023.xlsx")
OUTPUT_DIR = Path("outputs_pln_ipn")
FIG_DIR = OUTPUT_DIR / "figuras"
TABLE_DIR = OUTPUT_DIR / "tablas"
MODEL_DIR = OUTPUT_DIR / "modelos"
DICT_DIR = OUTPUT_DIR / "diccionario"

for folder in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR, DICT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Análisis avanzados activados: embeddings semánticos, UMAP y BERTopic.
# Si el modelo ya fue descargado, se reutiliza la caché local de embeddings.
RUN_EMBEDDINGS = True
RUN_BERTOPIC = True

print("Carpeta de trabajo:", Path.cwd())
print("Archivo de datos esperado:", DATA_PATH.resolve())
print("Embeddings disponibles:", HAS_EMBEDDINGS, "| BERTopic disponible:", HAS_BERTOPIC)


## 2. Carga de datos

Se cargan las hojas `Iniciativas` y `Capítulos` del archivo Excel. Luego se normalizan los nombres de columnas con `strip()` para corregir espacios finales, especialmente en `Nº de apoyos`.

**Qué se busca:** confirmar que la base contiene las columnas necesarias para cruzar texto con metadatos institucionales y de apoyo ciudadano.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {DATA_PATH}. Coloque el archivo en data/raw/IPN_19_07_2023.xlsx dentro del repositorio."
    )

iniciativas = pd.read_excel(DATA_PATH, sheet_name="Iniciativas")
capitulos = pd.read_excel(DATA_PATH, sheet_name="Capítulos")
iniciativas.columns = iniciativas.columns.str.strip()
capitulos.columns = capitulos.columns.str.strip()

expected_cols = [
    "Título", "Descripción", "Capítulo", "Nº Capítulo_iniciativa", "Nº Comisión", "Comisión",
    "Tipo", "Organización", "Pertenece a Organización", "Nº de apoyos", "Tramo Nº de apoyos",
    "Más de 10mil apoyos", "Más de 100 apoyos"
]
missing = [c for c in expected_cols if c not in iniciativas.columns]
if missing:
    raise ValueError(f"Faltan columnas esperadas: {missing}")

iniciativas["Tramo Nº de apoyos"] = iniciativas["Tramo Nº de apoyos"].replace(
    {"Entre 5000 y 9999 años": "Entre 5000 y 9999 apoyos"}
)

print("Dimensiones iniciativas:", iniciativas.shape)
print("Dimensiones capítulos:", capitulos.shape)
display(iniciativas.head(3))
display(capitulos.head(3))


**Interpretación.** La carga valida que el análisis puede combinar textos (`Título` y `Descripción`) con variables sustantivas como comisión, capítulo, tipo, organización y número de apoyos. Esto es importante porque la sistematización no se limita a contar palabras: busca relacionar patrones textuales con estructura institucional y adhesión ciudadana.


## 3. Auditoría de calidad de datos

Antes de aplicar técnicas de PLN, se revisa la calidad mínima de la base: nulos, duplicados, extensión textual y presencia de organización. Esta auditoría permite saber si los textos son analizables y qué limitaciones podrían afectar la interpretación.


In [ ]:
auditoria = {
    "registros": len(iniciativas),
    "nulos_titulo": int(iniciativas["Título"].isna().sum()),
    "nulos_descripcion": int(iniciativas["Descripción"].isna().sum()),
    "duplicados_titulo_descripcion": int(iniciativas.duplicated(["Título", "Descripción"]).sum()),
    "porcentaje_con_organizacion": round(100 * iniciativas["Organización"].notna().mean(), 2),
}

iniciativas["titulo_caracteres"] = iniciativas["Título"].fillna("").astype(str).str.len()
iniciativas["descripcion_caracteres"] = iniciativas["Descripción"].fillna("").astype(str).str.len()
iniciativas["titulo_palabras"] = iniciativas["Título"].fillna("").astype(str).str.split().str.len()
iniciativas["descripcion_palabras"] = iniciativas["Descripción"].fillna("").astype(str).str.split().str.len()

auditoria.update({
    "promedio_caracteres_titulo": round(iniciativas["titulo_caracteres"].mean(), 2),
    "promedio_caracteres_descripcion": round(iniciativas["descripcion_caracteres"].mean(), 2),
    "promedio_palabras_titulo": round(iniciativas["titulo_palabras"].mean(), 2),
    "promedio_palabras_descripcion": round(iniciativas["descripcion_palabras"].mean(), 2),
})

auditoria_df = pd.DataFrame([{"indicador": k, "valor": v} for k, v in auditoria.items()])
nulos_df = iniciativas.isna().sum().reset_index()
nulos_df.columns = ["columna", "nulos"]
nulos_df["porcentaje"] = (100 * nulos_df["nulos"] / len(iniciativas)).round(2)

auditoria_df.to_csv(TABLE_DIR / "auditoria_base.csv", index=False)
nulos_df.to_csv(TABLE_DIR / "nulos_por_columna.csv", index=False)

display(auditoria_df)
display(nulos_df.sort_values("nulos", ascending=False).head(10))


**Interpretación.** La auditoría muestra si la base es suficientemente completa para análisis textual. Una proporción baja de nulos en título y descripción permite aplicar métodos de frecuencia, TF-IDF y diccionario con bajo riesgo de pérdida de información. La presencia o ausencia de organización se conserva como variable analítica porque puede relacionarse con capacidad de movilización o formalización de propuestas.


## 4. Descripción general de la base

Se describen las iniciativas por comisión, capítulo, tipo, tramo de apoyos y presencia de organización. También se identifican las iniciativas con más apoyos. Esta sección responde qué tan distribuida o concentrada está la participación antes de entrar al contenido textual.


In [ ]:
def guardar_tabla(df, nombre):
    path = TABLE_DIR / nombre
    df.to_csv(path, index=False)
    return path

def guardar_figura(nombre):
    path = FIG_DIR / nombre
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

tabla_comision = iniciativas["Comisión"].value_counts().rename_axis("Comisión").reset_index(name="iniciativas")
tabla_capitulo = iniciativas["Capítulo"].value_counts().rename_axis("Capítulo").reset_index(name="iniciativas")
tabla_tipo = iniciativas["Tipo"].value_counts().rename_axis("Tipo").reset_index(name="iniciativas")
tabla_tramo = iniciativas["Tramo Nº de apoyos"].value_counts().rename_axis("Tramo Nº de apoyos").reset_index(name="iniciativas")
tabla_org = iniciativas["Organización"].notna().map({True: "Con organización", False: "Sin organización"}).value_counts().rename_axis("organizacion").reset_index(name="iniciativas")
top_apoyos = iniciativas.sort_values("Nº de apoyos", ascending=False).head(10)[
    ["Nº", "Título", "Comisión", "Capítulo", "Nº de apoyos", "Más de 10mil apoyos"]
]

for df, name in [
    (tabla_comision, "iniciativas_por_comision.csv"),
    (tabla_capitulo, "iniciativas_por_capitulo.csv"),
    (tabla_tipo, "iniciativas_por_tipo.csv"),
    (tabla_tramo, "iniciativas_por_tramo_apoyos.csv"),
    (tabla_org, "iniciativas_con_sin_organizacion.csv"),
    (top_apoyos, "top_10_iniciativas_apoyos.csv"),
]:
    guardar_tabla(df, name)

display(tabla_comision)
display(tabla_capitulo.head(15))
display(top_apoyos)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=tabla_comision.sort_values("iniciativas"), y="Comisión", x="iniciativas", ax=axes[0], color="#4C78A8")
axes[0].set_title("Iniciativas por comisión")
axes[0].set_xlabel("Número de iniciativas")
axes[0].set_ylabel("")

sns.barplot(data=tabla_capitulo.sort_values("iniciativas"), y="Capítulo", x="iniciativas", ax=axes[1], color="#59A14F")
axes[1].set_title("Iniciativas por capítulo")
axes[1].set_xlabel("Número de iniciativas")
axes[1].set_ylabel("")
guardar_figura("01_iniciativas_por_comision_capitulo.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(iniciativas["Nº de apoyos"], bins=50, ax=axes[0], color="#F28E2B")
axes[0].set_title("Distribución de apoyos")
axes[0].set_xlabel("Nº de apoyos")
sns.histplot(np.log1p(iniciativas["Nº de apoyos"]), bins=50, ax=axes[1], color="#B07AA1")
axes[1].set_title("Apoyos en escala logarítmica")
axes[1].set_xlabel("log(1 + apoyos)")
sns.barplot(data=tabla_tipo, y="Tipo", x="iniciativas", ax=axes[2], color="#E15759")
axes[2].set_title("Tipo de iniciativa")
axes[2].set_xlabel("Número de iniciativas")
axes[2].set_ylabel("")
guardar_figura("02_apoyos_y_tipo.png")


**Interpretación.** Las visualizaciones permiten observar concentración institucional y asimetría en apoyos. Esta distinción será relevante durante el análisis textual: un tema puede ser frecuente porque aparece en muchas iniciativas, pero no necesariamente ser el que concentra más apoyos ciudadanos.


## 5. Preparación textual

Se construyen tres versiones del texto: `texto_completo`, `texto_natural` y `texto_limpio`. La primera une título y descripción; la segunda conserva una versión legible para embeddings y ejemplos; la tercera normaliza el texto para conteos, bigramas y TF-IDF.

**Criterio:** no se eliminan globalmente palabras conceptuales como `estado`, `derecho`, `familia`, `educación`, `salud` o `seguridad`, porque son objetos sustantivos de análisis.


In [ ]:
def remover_tildes(texto):
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", str(texto))
        if not unicodedata.combining(ch)
    )

def normalizar_texto(texto, quitar_numeros=False):
    texto = "" if pd.isna(texto) else str(texto)
    texto = remover_tildes(texto.lower())
    texto = re.sub(r"https?://\S+", " ", texto)
    if quitar_numeros:
        texto = re.sub(r"\d+", " ", texto)
    texto = re.sub(r"[^a-zñü0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

stopwords_dominio = {
    "iniciativa", "iniciativas", "constitucion", "constitucional", "capitulo",
    "articulo", "articulos", "propuesta", "popular", "populares", "proceso",
    "norma", "normas", "chile"
}
stopwords_es = set(STOPWORDS_NLTK) | stopwords_dominio

iniciativas["texto_completo"] = (
    iniciativas["Título"].fillna("").astype(str).str.strip()
    + ". "
    + iniciativas["Descripción"].fillna("").astype(str).str.strip()
)
iniciativas["texto_natural"] = iniciativas["texto_completo"].str.replace(r"\s+", " ", regex=True).str.strip()
iniciativas["texto_limpio"] = iniciativas["texto_completo"].map(lambda x: normalizar_texto(x, quitar_numeros=True))
iniciativas["tokens_limpios"] = iniciativas["texto_limpio"].map(
    lambda txt: [t for t in txt.split() if len(t) >= 3 and t not in stopwords_es]
)
iniciativas["n_tokens_limpios"] = iniciativas["tokens_limpios"].map(len)

display(iniciativas[["Título", "texto_limpio", "tokens_limpios"]].head(3))


**Interpretación.** La separación entre texto natural y texto limpio evita mezclar objetivos: los conteos requieren normalización fuerte, mientras que embeddings, KWIC y lectura humana necesitan conservar mayor naturalidad textual. Esta decisión aumenta la trazabilidad del análisis.


## 6. Identificación de nominales y sintagmas nominales candidatos

Antes de pasar a frecuencias generales, se agrega una etapa de identificación de nominales. En PLN, los nominales son expresiones centradas en sustantivos o nombres, por ejemplo `medio ambiente`, `derecho a la vida`, `poder judicial`, `educación pública` o `seguridad social`. Estas expresiones suelen ser más informativas que palabras aisladas porque condensan temas, actores o instituciones.

En un flujo ideal, los sintagmas nominales se extraen con análisis morfosintáctico y dependencias. Para mantener el notebook ejecutable sin depender de modelos externos de spaCy, se usa una aproximación auditable: se generan candidatos de 1 a 3 palabras con `CountVectorizer`, se filtran stopwords y verbos funcionales frecuentes, y se retienen expresiones con frecuencia mínima. La limitación es que se trata de **nominales candidatos**, no de una anotación gramatical perfecta.


In [ ]:
stopwords_nominales = stopwords_es | {
    "debe", "deben", "ser", "sera", "sean", "puede", "pueden", "hacer", "tener",
    "forma", "manera", "respecto", "mediante", "establecer", "establece", "garantizar",
    "art", "eliminar", "nueva", "rectificada", "criterios", "redaccion", "lectura"
}

def extraer_nominales_candidatos(textos, min_df=2, top_n=60):
    vectorizer = CountVectorizer(
        preprocessor=lambda x: normalizar_texto(x, quitar_numeros=True),
        stop_words=list(stopwords_nominales),
        ngram_range=(1, 3),
        min_df=min_df,
    )
    X = vectorizer.fit_transform(textos)
    terms = np.array(vectorizer.get_feature_names_out())
    counts = np.asarray(X.sum(axis=0)).ravel()

    rows = []
    for term, freq in zip(terms, counts):
        partes = term.split()
        # Filtro simple: evita candidatos demasiado genéricos y premia expresiones compuestas.
        if len(partes) == 1 and len(term) < 4:
            continue
        if all(p in stopwords_nominales for p in partes):
            continue
        rows.append({
            "nominal_candidato": term,
            "n_palabras": len(partes),
            "frecuencia": int(freq),
        })
    out = pd.DataFrame(rows).sort_values(["frecuencia", "n_palabras"], ascending=[False, False])
    return out.head(top_n), vectorizer, X

nominales_globales, nominal_vectorizer, X_nominal = extraer_nominales_candidatos(iniciativas["texto_completo"], min_df=2, top_n=500)
nominales_globales.to_csv(TABLE_DIR / "nominales_candidatos_globales.csv", index=False)
nominales_compuestos = nominales_globales[nominales_globales["n_palabras"].ge(2)].head(40)
nominales_compuestos.to_csv(TABLE_DIR / "nominales_compuestos_globales.csv", index=False)
display(nominales_globales.head(20))
display(nominales_compuestos.head(20))

nominales_por_comision = []
for comision, grupo in iniciativas.groupby("Comisión"):
    tmp, _, _ = extraer_nominales_candidatos(grupo["texto_completo"], min_df=2, top_n=80)
    tmp["Comisión"] = comision
    nominales_por_comision.append(tmp)
nominales_por_comision = pd.concat(nominales_por_comision, ignore_index=True)
nominales_por_comision.to_csv(TABLE_DIR / "nominales_candidatos_por_comision.csv", index=False)
display(nominales_por_comision.groupby("Comisión").head(8))

top_nominales = nominales_compuestos.sort_values("frecuencia", ascending=False).head(25)
plt.figure(figsize=(10, 6))
sns.barplot(data=top_nominales, y="nominal_candidato", x="frecuencia", color="#76B7B2")
plt.title("Nominales compuestos candidatos más frecuentes")
plt.xlabel("Frecuencia")
plt.ylabel("")
guardar_figura("03b_nominales_candidatos_globales.png")

# Etiquetado ligero por documento: nominales candidatos presentes en cada iniciativa.
top_nominal_terms = nominales_globales["nominal_candidato"].head(80).tolist()
nominal_patterns = {
    term: re.compile(r"\b" + re.escape(term).replace(r"\ ", r"\s+") + r"\b")
    for term in top_nominal_terms
}

def nominales_en_documento(texto, max_items=8):
    texto_norm = normalizar_texto(texto, quitar_numeros=True)
    encontrados = [term for term, pat in nominal_patterns.items() if pat.search(texto_norm)]
    return encontrados[:max_items]

iniciativas["nominales_candidatos"] = iniciativas["texto_completo"].map(nominales_en_documento)
iniciativas["n_nominales_candidatos"] = iniciativas["nominales_candidatos"].map(len)
display(iniciativas[["Título", "nominales_candidatos"]].head(8))


**Interpretación.** Los nominales candidatos permiten una lectura intermedia entre tokens aislados y modelos de tópicos. Son especialmente útiles para este corpus porque muchas demandas aparecen como expresiones compuestas: instituciones, derechos, grupos sociales o bienes públicos. La limitación es que el método identifica candidatos por patrones estadísticos, por lo que algunos resultados requieren revisión humana.


## 6.1. Diagrama de visualización de componentes gramaticales

Como complemento creativo a los nominales, se construye un diagrama de componentes gramaticales y funcionales del corpus. La idea es representar no solo palabras frecuentes, sino tipos de piezas lingüísticas que aparecen en las iniciativas:

- **Nominales compuestos:** expresiones como `medio ambiente`, `poder judicial` o `derecho vivienda`.
- **Acciones normativas:** verbos o formas de acción como reconocer, proteger, garantizar, modificar o regular.
- **Instituciones y actores:** Estado, Congreso, Presidente, Poder Judicial, municipalidades, regiones.
- **Referencias normativas:** artículo, ley, norma, capítulo.
- **Cifras:** números, porcentajes, años u otras referencias cuantitativas.
- **Atributos:** adjetivos frecuentes en lenguaje constitucional, como público, social, nacional, regional o fundamental.

Esta visualización no pretende reemplazar un análisis morfosintáctico formal. Es una aproximación explicable para mostrar cómo se organizan los componentes textuales más relevantes del corpus.


In [ ]:
componentes_def = {
    "Nominales compuestos": nominales_compuestos["nominal_candidato"].head(18).tolist(),
    "Acciones normativas": [
        "reconocer", "proteger", "garantizar", "regular", "modificar", "agregar",
        "eliminar", "asegurar", "establecer", "consagrar", "prohibir", "promover"
    ],
    "Instituciones y actores": [
        "estado", "congreso", "presidente", "gobierno", "poder judicial", "contraloria",
        "ministerio publico", "municipalidad", "municipalidades", "region", "regiones"
    ],
    "Referencias normativas": [
        "articulo", "articulos", "ley", "norma", "capitulo", "constitucion", "reglamento"
    ],
    "Cifras": [r"\b\d+([.,]\d+)?\s*(%|por ciento|anos|dias|meses|uf|pesos)?\b"],
    "Atributos constitucionales": [
        "publico", "publica", "social", "nacional", "regional", "fundamental",
        "constitucional", "ambiental", "cultural", "economico", "politico", "juridico"
    ],
}

def contar_patron_en_textos(textos, patron):
    return sum(len(re.findall(patron, txt)) for txt in textos)

componente_rows = []
for componente, terminos in componentes_def.items():
    for termino in terminos:
        if componente == "Cifras":
            patron = termino
            etiqueta = "referencias numéricas"
        else:
            etiqueta = termino
            patron = r"\b" + re.escape(normalizar_texto(termino, quitar_numeros=False)).replace(r"\ ", r"\s+") + r"\b"
        textos_fuente = iniciativas["texto_natural"] if componente == "Cifras" else iniciativas["texto_limpio"]
        frecuencia = contar_patron_en_textos(textos_fuente, patron)
        if frecuencia > 0:
            componente_rows.append({
                "componente": componente,
                "elemento": etiqueta,
                "frecuencia": int(frecuencia),
            })

componentes_df = pd.DataFrame(componente_rows).sort_values(["componente", "frecuencia"], ascending=[True, False])
componentes_df.to_csv(TABLE_DIR / "componentes_gramaticales_detalle.csv", index=False)
display(componentes_df.groupby("componente").head(8))

componentes_resumen = (
    componentes_df.groupby("componente")["frecuencia"]
    .sum()
    .reset_index()
    .sort_values("frecuencia", ascending=False)
)
componentes_resumen.to_csv(TABLE_DIR / "componentes_gramaticales_resumen.csv", index=False)

comp_por_comision_rows = []
for comision, grupo in iniciativas.groupby("Comisión"):
    row = {"Comisión": comision, "n": len(grupo)}
    for componente, terminos in componentes_def.items():
        total = 0
        for termino in terminos:
            if componente == "Cifras":
                patron = termino
            else:
                patron = r"\b" + re.escape(normalizar_texto(termino, quitar_numeros=False)).replace(r"\ ", r"\s+") + r"\b"
            textos_grupo = grupo["texto_natural"] if componente == "Cifras" else grupo["texto_limpio"]
            total += contar_patron_en_textos(textos_grupo, patron)
        row[componente] = total
    comp_por_comision_rows.append(row)
componentes_comision = pd.DataFrame(comp_por_comision_rows)
componentes_comision.to_csv(TABLE_DIR / "componentes_gramaticales_por_comision.csv", index=False)

# Diagrama de red: corpus -> componente -> elementos más frecuentes.
G_comp = nx.Graph()
root = "Corpus IPN"
G_comp.add_node(root, tipo="raiz")
for componente in componentes_resumen["componente"]:
    G_comp.add_node(componente, tipo="componente")
    G_comp.add_edge(root, componente, weight=componentes_resumen.loc[componentes_resumen["componente"].eq(componente), "frecuencia"].iloc[0])
    top_items = componentes_df[componentes_df["componente"].eq(componente)].head(5)
    for _, item in top_items.iterrows():
        label = item["elemento"]
        G_comp.add_node(label, tipo="elemento")
        G_comp.add_edge(componente, label, weight=item["frecuencia"])

color_map = {"raiz": "#0B2545", "componente": "#76B7B2", "elemento": "#E8EEF5"}
node_colors = [color_map.get(G_comp.nodes[n].get("tipo"), "#CCCCCC") for n in G_comp.nodes]
node_sizes = [2200 if G_comp.nodes[n].get("tipo") == "raiz" else 1500 if G_comp.nodes[n].get("tipo") == "componente" else 850 for n in G_comp.nodes]
plt.figure(figsize=(13, 9))
pos = nx.spring_layout(G_comp, seed=RANDOM_STATE, k=0.55)
weights = [G_comp[u][v]["weight"] for u, v in G_comp.edges]
nx.draw_networkx_nodes(G_comp, pos, node_color=node_colors, node_size=node_sizes, edgecolors="#333333", linewidths=0.8)
nx.draw_networkx_edges(G_comp, pos, width=[1 + 4*w/max(weights) for w in weights], alpha=0.35, edge_color="#555555")
nx.draw_networkx_labels(G_comp, pos, font_size=8)
plt.title("Diagrama de componentes gramaticales y funcionales del corpus")
plt.axis("off")
guardar_figura("03c_diagrama_componentes_gramaticales.png")

heat_comp = componentes_comision.set_index("Comisión").drop(columns=["n"])
plt.figure(figsize=(11, 4.8))
sns.heatmap(heat_comp, annot=True, fmt=".0f", cmap="Purples")
plt.title("Componentes gramaticales por comisión")
plt.xlabel("Componente")
plt.ylabel("Comisión")
guardar_figura("03d_componentes_gramaticales_por_comision.png")


**Interpretación.** El diagrama muestra que el corpus no solo contiene temas, sino tipos de piezas lingüísticas: nominales que nombran asuntos, acciones normativas que expresan qué se quiere hacer, instituciones que ubican responsabilidades y referencias normativas que conectan las propuestas con lenguaje constitucional. Esta lectura ayuda a explicar la estructura gramatical del material antes de pasar a modelos más abstractos.


## 7. Frecuencia de tokens

La frecuencia de tokens entrega una primera aproximación a los términos más presentes en el corpus. Su principal ventaja es la transparencia; su principal limitación es que no distingue por sí sola contexto, sentido ni relevancia institucional.


In [ ]:
def top_tokens(series_tokens, n=30):
    counter = Counter()
    for tokens in series_tokens:
        counter.update(tokens)
    return pd.DataFrame(counter.most_common(n), columns=["termino", "frecuencia"])

top_tokens_global = top_tokens(iniciativas["tokens_limpios"], 30)
top_tokens_global.to_csv(TABLE_DIR / "top_30_tokens_globales.csv", index=False)
display(top_tokens_global)

plt.figure(figsize=(9, 7))
sns.barplot(data=top_tokens_global, y="termino", x="frecuencia", color="#4C78A8")
plt.title("Top 30 tokens globales")
plt.xlabel("Frecuencia")
plt.ylabel("")
guardar_figura("03_top_30_tokens_globales.png")

top_tokens_comision = []
for comision, grupo in iniciativas.groupby("Comisión"):
    tmp = top_tokens(grupo["tokens_limpios"], 20)
    tmp["Comisión"] = comision
    top_tokens_comision.append(tmp)
top_tokens_comision = pd.concat(top_tokens_comision, ignore_index=True)
top_tokens_comision.to_csv(TABLE_DIR / "top_tokens_por_comision.csv", index=False)
display(top_tokens_comision.groupby("Comisión").head(8))


**Interpretación.** Los tokens frecuentes muestran los ejes semánticos dominantes, pero deben leerse con cautela. Para reducir el riesgo de una lectura superficial se complementan con bigramas, TF-IDF, diccionario temático y ejemplos KWIC.


## 8. Bigramas

Los bigramas permiten observar expresiones de dos palabras que entregan más contexto que los tokens aislados. Esta técnica ayuda a diferenciar, por ejemplo, una palabra general como `medio` de una expresión constitucionalmente relevante como `medio ambiente`.


In [ ]:
def calcular_ngrams(textos, ngram_range=(2, 2), min_df=2, top_n=30):
    vectorizer = CountVectorizer(
        preprocessor=lambda x: normalizar_texto(x, quitar_numeros=True),
        stop_words=list(stopwords_es),
        ngram_range=ngram_range,
        min_df=min_df,
    )
    X = vectorizer.fit_transform(textos)
    terms = np.array(vectorizer.get_feature_names_out())
    counts = np.asarray(X.sum(axis=0)).ravel()
    order = counts.argsort()[::-1][:top_n]
    return pd.DataFrame({"ngram": terms[order], "frecuencia": counts[order]})

top_bigramas_global = calcular_ngrams(iniciativas["texto_completo"], top_n=30)
top_bigramas_global.to_csv(TABLE_DIR / "top_30_bigramas_globales.csv", index=False)
display(top_bigramas_global)

plt.figure(figsize=(9, 7))
sns.barplot(data=top_bigramas_global, y="ngram", x="frecuencia", color="#E15759")
plt.title("Top 30 bigramas globales")
plt.xlabel("Frecuencia")
plt.ylabel("")
guardar_figura("04_top_30_bigramas_globales.png")

bigramas_comision = []
for comision, grupo in iniciativas.groupby("Comisión"):
    tmp = calcular_ngrams(grupo["texto_completo"], top_n=15)
    tmp["Comisión"] = comision
    bigramas_comision.append(tmp)
bigramas_comision = pd.concat(bigramas_comision, ignore_index=True)
bigramas_comision.to_csv(TABLE_DIR / "bigramas_por_comision.csv", index=False)
display(bigramas_comision.groupby("Comisión").head(6))


**Interpretación.** Los bigramas aportan contexto semántico y permiten detectar expresiones institucionales, sociales o valóricas que no aparecen con claridad en conteos de palabras individuales.


## 9. TF-IDF global y por grupos

TF-IDF pondera términos según su frecuencia en documentos y su capacidad de distinguir textos. Aquí se usa para identificar términos característicos por comisión y capítulo, no solo términos frecuentes en toda la base.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    preprocessor=lambda x: normalizar_texto(x, quitar_numeros=True),
    stop_words=list(stopwords_es),
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    max_features=3000,
)
X_tfidf = tfidf_vectorizer.fit_transform(iniciativas["texto_completo"])
tfidf_terms = np.array(tfidf_vectorizer.get_feature_names_out())
tfidf_global_scores = np.asarray(X_tfidf.mean(axis=0)).ravel()
tfidf_global = pd.DataFrame({
    "termino": tfidf_terms,
    "tfidf_promedio": tfidf_global_scores,
}).sort_values("tfidf_promedio", ascending=False).head(30)
tfidf_global.to_csv(TABLE_DIR / "tfidf_global_top_30.csv", index=False)
display(tfidf_global)

tfidf_comision_rows = []
for comision in iniciativas["Comisión"].unique():
    idx = iniciativas["Comisión"].eq(comision).values
    scores = np.asarray(X_tfidf[idx].mean(axis=0)).ravel()
    order = scores.argsort()[::-1][:15]
    for pos in order:
        tfidf_comision_rows.append({
            "Comisión": comision,
            "termino": tfidf_terms[pos],
            "tfidf_promedio": scores[pos],
        })
tfidf_comision = pd.DataFrame(tfidf_comision_rows)
tfidf_comision.to_csv(TABLE_DIR / "tfidf_por_comision.csv", index=False)
display(tfidf_comision.groupby("Comisión").head(8))

top_heat_terms = tfidf_comision.groupby("Comisión").head(8)["termino"].unique().tolist()
heat_data = []
for comision in iniciativas["Comisión"].unique():
    idx = iniciativas["Comisión"].eq(comision).values
    scores = np.asarray(X_tfidf[idx].mean(axis=0)).ravel()
    score_map = dict(zip(tfidf_terms, scores))
    row = {"Comisión": comision}
    row.update({term: score_map.get(term, 0) for term in top_heat_terms})
    heat_data.append(row)
tfidf_heat = pd.DataFrame(heat_data).set_index("Comisión")
tfidf_heat.to_csv(TABLE_DIR / "tfidf_heatmap_comision.csv")

plt.figure(figsize=(13, 4.8))
sns.heatmap(tfidf_heat, cmap="Blues")
plt.title("Términos distintivos TF-IDF por comisión")
plt.xlabel("Término")
plt.ylabel("Comisión")
guardar_figura("05_heatmap_tfidf_comision.png")


**Interpretación.** TF-IDF ayuda a diferenciar frecuencia general de especificidad temática. Si un término aparece mucho en una comisión y poco en otras, puede funcionar como señal distintiva de ese grupo institucional.


## 10. Clustering separado por comisión

Dado que la base ya está clasificada institucionalmente por comisión, un clustering global tendería a redescubrir esa misma estructura. Por eso se propone un clustering **dentro de cada comisión**, usando TF-IDF y KMeans. La pregunta cambia: no se busca saber qué comisión es distinta, sino qué subgrupos temáticos aparecen al interior de cada comisión.

El número de clusters se define de forma adaptativa según el tamaño de cada comisión. Los clusters no son categorías oficiales ni sustituyen el diccionario temático; funcionan como una exploración no supervisada para detectar subtemas internos.


In [ ]:
def elegir_k_cluster(n_docs):
    if n_docs < 30:
        return 2
    if n_docs < 120:
        return 3
    if n_docs < 300:
        return 4
    return 5

iniciativas["cluster_interno_comision"] = ""
iniciativas["cluster_interno_num"] = np.nan
cluster_rows = []
cluster_doc_rows = []

for comision, grupo in iniciativas.groupby("Comisión"):
    n_docs = len(grupo)
    k = min(elegir_k_cluster(n_docs), n_docs - 1)
    if k < 2:
        continue

    vectorizer_cluster = TfidfVectorizer(
        preprocessor=lambda x: normalizar_texto(x, quitar_numeros=True),
        stop_words=list(stopwords_nominales),
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.85,
        max_features=1200,
    )
    X_cluster = vectorizer_cluster.fit_transform(grupo["texto_completo"])
    terms_cluster = np.array(vectorizer_cluster.get_feature_names_out())

    if X_cluster.shape[1] < k:
        k = max(2, min(k, X_cluster.shape[1]))

    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X_cluster)

    silhouette = np.nan
    try:
        if len(set(labels)) > 1 and n_docs > k:
            sample_size = min(300, n_docs)
            silhouette = silhouette_score(X_cluster, labels, sample_size=sample_size, random_state=RANDOM_STATE)
    except Exception:
        silhouette = np.nan

    for cluster_id in range(k):
        mask = labels == cluster_id
        size = int(mask.sum())
        centroid = km.cluster_centers_[cluster_id]
        top_idx = centroid.argsort()[::-1][:12]
        top_terms = ", ".join(terms_cluster[top_idx])
        cluster_label = f"C{cluster_id + 1}"
        cluster_rows.append({
            "Comisión": comision,
            "cluster": cluster_label,
            "n_iniciativas": size,
            "porcentaje_comision": round(100 * size / n_docs, 2),
            "k_comision": k,
            "silhouette_comision": round(float(silhouette), 3) if not pd.isna(silhouette) else np.nan,
            "terminos_representativos": top_terms,
        })

    for idx, label in zip(grupo.index, labels):
        cluster_label = f"C{label + 1}"
        iniciativas.loc[idx, "cluster_interno_comision"] = cluster_label
        iniciativas.loc[idx, "cluster_interno_num"] = int(label + 1)
        cluster_doc_rows.append({
            "Nº": iniciativas.loc[idx, "Nº"],
            "Título": iniciativas.loc[idx, "Título"],
            "Comisión": comision,
            "cluster": cluster_label,
            "Nº de apoyos": iniciativas.loc[idx, "Nº de apoyos"],
            "Capítulo": iniciativas.loc[idx, "Capítulo"],
        })

clusters_resumen = pd.DataFrame(cluster_rows)
clusters_documentos = pd.DataFrame(cluster_doc_rows)
clusters_resumen.to_csv(TABLE_DIR / "clusters_por_comision_resumen.csv", index=False)
clusters_documentos.to_csv(TABLE_DIR / "documentos_clusters_por_comision.csv", index=False)
display(clusters_resumen)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=clusters_resumen,
    y="Comisión",
    x="porcentaje_comision",
    hue="cluster",
)
plt.title("Tamaño relativo de clusters internos por comisión")
plt.xlabel("% dentro de la comisión")
plt.ylabel("")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
guardar_figura("05b_clusters_internos_por_comision.png")

print("Ejemplos de documentos por cluster")
for _, row in clusters_resumen.iterrows():
    subset = clusters_documentos[
        clusters_documentos["Comisión"].eq(row["Comisión"])
        & clusters_documentos["cluster"].eq(row["cluster"])
    ].sort_values("Nº de apoyos", ascending=False).head(3)
    print(f"\n{row['Comisión']} | {row['cluster']} | {row['terminos_representativos']}")
    display(subset[["Nº", "Título", "Nº de apoyos"]])


**Interpretación.** Sí, es posible hacer clustering por comisión y es metodológicamente más defendible que un clustering global cuando ya existe una clasificación institucional. La lectura correcta es interna: cada cluster representa un subgrupo temático aproximado dentro de su comisión. La calidad no debe evaluarse solo por métricas como silhouette, sino también por la coherencia de términos representativos y ejemplos de iniciativas.


## 11. Construcción del diccionario temático constitucional

Se construye un diccionario temático propio, diseñado para el dominio constitucional chileno y las Iniciativas Populares 2023. Se opta por un diccionario propio porque permite trazabilidad, explicación metodológica, auditoría por terceros y reutilización en otros corpus de participación ciudadana.

Las categorías se definen a partir de temas esperables en debates constitucionales: derechos sociales, medio ambiente, sistema político, justicia, familia, seguridad, participación, descentralización, economía, pueblos indígenas, educación, salud, cuidados, agua, niñez, género, religión, trabajo, probidad y ciencia. El diccionario no se entiende como verdad absoluta, sino como herramienta supervisada para organizar evidencia textual.


In [ ]:
categorias = {
    "Derechos sociales": ["derechos sociales", "vivienda", "seguridad social", "prevision", "pensiones", "dignidad", "bienestar"],
    "Medio ambiente": ["medio ambiente", "cambio climatico", "naturaleza", "ambiental", "biodiversidad", "proteccion animal"],
    "Sistema político": ["presidente", "congreso", "diputados", "senadores", "partidos politicos", "eleccion", "voto"],
    "Justicia e instituciones": ["poder judicial", "tribunales", "jueces", "contraloria", "ministerio publico", "servicio electoral"],
    "Familia y vida": ["familia", "derecho a la vida", "vida", "nacer", "concepcion", "matrimonio"],
    "Seguridad y orden público": ["seguridad", "orden publico", "delincuencia", "carabineros", "policia", "terrorismo", "violencia"],
    "Participación ciudadana": ["participacion ciudadana", "plebiscito", "consulta", "iniciativa popular", "cabildos", "sociedad civil"],
    "Descentralización y territorio": ["region", "regional", "gobierno regional", "municipalidad", "municipalidades", "territorio", "comuna"],
    "Economía, propiedad y emprendimiento": ["propiedad", "emprendimiento", "empresa", "economia", "tributos", "impuestos", "libertad economica"],
    "Pueblos indígenas": ["pueblos indigenas", "indigena", "mapuche", "originarios", "plurinacional", "intercultural"],
    "Educación": ["educacion", "educacion publica", "ensenanza", "universidad", "colegio", "gratuidad", "calidad"],
    "Salud": ["salud", "salud mental", "hospital", "atencion primaria", "enfermedad", "medicamentos"],
    "Cuidados": ["cuidados", "cuidado", "personas mayores", "dependencia", "discapacidad", "cuidadoras"],
    "Agua y recursos naturales": ["agua", "recursos naturales", "mineria", "glaciares", "cuencas", "mar"],
    "Niñez, adolescencia y personas mayores": ["ninez", "ninos", "ninas", "adolescentes", "personas mayores", "adulto mayor"],
    "Género e igualdad": ["genero", "igualdad", "mujeres", "discriminacion", "paridad", "equidad"],
    "Religión, conciencia y valores": ["religion", "libertad religiosa", "conciencia", "valores", "iglesia", "culto"],
    "Trabajo y seguridad social": ["trabajo", "trabajadores", "sindicato", "laboral", "seguridad social", "pensiones"],
    "Probidad, transparencia y control público": ["probidad", "transparencia", "corrupcion", "control publico", "fiscalizacion", "rendicion"],
    "Ciencia, tecnología e innovación": ["ciencia", "tecnologia", "innovacion", "investigacion", "digital", "datos"],
}

descripciones = {
    cat: f"Categoría destinada a identificar menciones relativas a {cat.lower()} en el debate constitucional."
    for cat in categorias
}

dict_rows = []
for categoria, terminos in categorias.items():
    for termino in terminos:
        variantes = sorted(set([termino, remover_tildes(termino), termino.replace(" ", r"\s+")]))
        dict_rows.append({
            "categoria": categoria,
            "subcategoria": "",
            "termino": termino,
            "variantes": "|".join(variantes),
            "descripcion_categoria": descripciones[categoria],
            "criterio_inclusion": "Término o expresión relevante para debates constitucionales y participación ciudadana.",
            "observaciones": "Diccionario inicial; requiere validación cualitativa y ajuste iterativo.",
        })

diccionario_df = pd.DataFrame(dict_rows)
diccionario_json = diccionario_df.to_dict(orient="records")
(DICT_DIR / "diccionario_tematico_constitucional.json").write_text(json.dumps(diccionario_json, ensure_ascii=False, indent=2), encoding="utf-8")
diccionario_df.to_csv(DICT_DIR / "diccionario_tematico_constitucional.csv", index=False)
diccionario_df.to_excel(DICT_DIR / "diccionario_tematico_constitucional.xlsx", index=False)

doc_diccionario = '''# Documentación del diccionario temático constitucional

Este diccionario fue construido específicamente para sistematizar las Iniciativas Populares del proceso constituyente 2023.

## Criterio metodológico
Las categorías agrupan campos sustantivos del debate constitucional chileno. Los términos fueron definidos como palabras o expresiones que permiten detectar menciones temáticas en textos ciudadanos breves.

## Campos
- categoria
- subcategoria
- termino
- variantes
- descripcion_categoria
- criterio_inclusion
- observaciones

## Limitaciones
El diccionario no detecta ironía, negación ni sentido contextual complejo. Su función es entregar trazabilidad e interpretabilidad, no reemplazar lectura cualitativa.
'''
(DICT_DIR / "documentacion_diccionario_tematico_constitucional.md").write_text(dedent(doc_diccionario), encoding="utf-8")

display(diccionario_df.head(20))
print("Entradas del diccionario:", len(diccionario_df))


In [ ]:
def compilar_patron_terminos(terminos):
    piezas = []
    for termino in terminos:
        t = remover_tildes(termino.lower())
        t = re.escape(t).replace(r"\ ", r"\s+")
        piezas.append(t)
    return re.compile(r"\b(" + "|".join(piezas) + r")\b", flags=re.IGNORECASE)

patrones_diccionario = {cat: compilar_patron_terminos(terms) for cat, terms in categorias.items()}

for categoria, patron in patrones_diccionario.items():
    col_menciones = f"dict_menciones__{categoria}"
    col_presencia = f"dict_presencia__{categoria}"
    iniciativas[col_menciones] = iniciativas["texto_limpio"].map(lambda txt: len(patron.findall(txt)))
    iniciativas[col_presencia] = iniciativas[col_menciones].gt(0).astype(int)
    iniciativas[f"dict_norm__{categoria}"] = iniciativas[col_menciones] / iniciativas["n_tokens_limpios"].replace(0, np.nan)

menciones_cols = [f"dict_menciones__{c}" for c in categorias]
presencia_cols = [f"dict_presencia__{c}" for c in categorias]
norm_cols = [f"dict_norm__{c}" for c in categorias]

resumen_diccionario = pd.DataFrame({
    "categoria": list(categorias.keys()),
    "menciones_brutas": [int(iniciativas[f"dict_menciones__{c}"].sum()) for c in categorias],
    "iniciativas_con_presencia": [int(iniciativas[f"dict_presencia__{c}"].sum()) for c in categorias],
    "prevalencia_pct": [round(100 * iniciativas[f"dict_presencia__{c}"].mean(), 2) for c in categorias],
    "menciones_norm_promedio": [round(iniciativas[f"dict_norm__{c}"].mean(skipna=True), 4) for c in categorias],
}).sort_values("prevalencia_pct", ascending=False)
resumen_diccionario.to_csv(TABLE_DIR / "resumen_diccionario_tematico.csv", index=False)
display(resumen_diccionario)

def prevalencia_por_grupo(grupo_col, nombre_archivo):
    rows = []
    for grupo, df_g in iniciativas.groupby(grupo_col, dropna=False):
        row = {grupo_col: grupo, "n": len(df_g)}
        for cat in categorias:
            row[cat] = round(100 * df_g[f"dict_presencia__{cat}"].mean(), 2)
        rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(TABLE_DIR / nombre_archivo, index=False)
    return out

prev_comision = prevalencia_por_grupo("Comisión", "prevalencia_temas_por_comision.csv")
prev_capitulo = prevalencia_por_grupo("Capítulo", "prevalencia_temas_por_capitulo.csv")
prev_tramo = prevalencia_por_grupo("Tramo Nº de apoyos", "prevalencia_temas_por_tramo_apoyos.csv")
iniciativas["grupo_mas_10mil"] = iniciativas["Más de 10mil apoyos"].map({1: "Más de 10 mil", 0: "Resto"})
iniciativas["grupo_organizacion"] = iniciativas["Organización"].notna().map({True: "Con organización", False: "Sin organización"})
prev_10mil = prevalencia_por_grupo("grupo_mas_10mil", "prevalencia_temas_mas_10mil_apoyos.csv")
prev_org = prevalencia_por_grupo("grupo_organizacion", "prevalencia_temas_organizacion.csv")

plt.figure(figsize=(14, 4.5))
sns.barplot(data=resumen_diccionario.head(15), y="categoria", x="prevalencia_pct", color="#4C78A8")
plt.title("Categorías temáticas más frecuentes según diccionario propio")
plt.xlabel("% de iniciativas con presencia")
plt.ylabel("")
guardar_figura("06_diccionario_categorias_frecuentes.png")

heat = prev_comision.set_index("Comisión").drop(columns=["n"])
plt.figure(figsize=(16, 4.8))
sns.heatmap(heat, annot=False, cmap="Greens")
plt.title("Prevalencia temática por comisión (%)")
plt.xlabel("Categoría")
plt.ylabel("Comisión")
guardar_figura("07_heatmap_diccionario_comision.png")

comp = prev_10mil.set_index("grupo_mas_10mil").drop(columns=["n"]).T.reset_index()
comp = comp.rename(columns={"index": "categoria"})
comp["diferencia_mas_10mil"] = comp.get("Más de 10 mil", 0) - comp.get("Resto", 0)
comp.to_csv(TABLE_DIR / "comparacion_diccionario_mas_10mil_vs_resto.csv", index=False)
display(comp.sort_values("diferencia_mas_10mil", ascending=False).head(10))

plot_comp = comp.sort_values("diferencia_mas_10mil", ascending=False).head(12)
plt.figure(figsize=(10, 5.5))
sns.barplot(data=plot_comp, y="categoria", x="diferencia_mas_10mil", color="#F28E2B")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Diferencia temática: más de 10 mil apoyos vs resto")
plt.xlabel("Diferencia en puntos porcentuales")
plt.ylabel("")
guardar_figura("08_diccionario_mas_10mil_vs_resto.png")

org_comp = prev_org.set_index("grupo_organizacion").drop(columns=["n"]).T.reset_index().rename(columns={"index": "categoria"})
org_comp.to_csv(TABLE_DIR / "comparacion_diccionario_organizacion.csv", index=False)


**Interpretación.** El diccionario propio permite pasar de conteos generales a categorías sustantivas auditables. Su ventaja principal es la trazabilidad: cada categoría está definida y puede revisarse, ampliarse o corregirse. Su limitación es que depende de decisiones del investigador y no captura automáticamente negaciones, ambigüedades ni contexto profundo.


## 12. Menciones específicas mediante expresiones regulares

Se analizan menciones a artículos, Estado, entidades y cifras. Estas reglas permiten responder preguntas sustantivas planteadas en la clase sin depender de un modelo externo de reconocimiento de entidades.


In [ ]:
patrones_especificos = {
    "articulos": r"\b(articulo|art\.?)\s*(\d+|[ivxlcdm]+)?",
    "estado": r"\b(estado|estatal|gobierno|ministerio|servicio publico)\b",
    "entidades": r"\b(congreso|presidente|gobierno|poder judicial|contraloria|ministerio publico|municipalidades|regiones|carabineros)\b",
    "cifras": r"\b\d+([.,]\d+)?\s*(%|por ciento|pesos|uf|anos|dias|meses)?\b",
}
for nombre, patron in patrones_especificos.items():
    iniciativas[f"regex_{nombre}"] = iniciativas["texto_limpio"].str.contains(patron, regex=True, na=False).astype(int)

regex_cols = [f"regex_{k}" for k in patrones_especificos]
regex_resumen = pd.DataFrame({
    "patron": list(patrones_especificos.keys()),
    "n_iniciativas": [int(iniciativas[f"regex_{k}"].sum()) for k in patrones_especificos],
    "porcentaje": [round(100 * iniciativas[f"regex_{k}"].mean(), 2) for k in patrones_especificos],
})
regex_resumen.to_csv(TABLE_DIR / "menciones_regex_resumen.csv", index=False)
display(regex_resumen)

regex_comision = iniciativas.groupby("Comisión")[regex_cols].mean().mul(100).round(2).reset_index()
regex_capitulo = iniciativas.groupby("Capítulo")[regex_cols].mean().mul(100).round(2).reset_index()
regex_comision.to_csv(TABLE_DIR / "menciones_regex_por_comision.csv", index=False)
regex_capitulo.to_csv(TABLE_DIR / "menciones_regex_por_capitulo.csv", index=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=regex_resumen, y="patron", x="porcentaje", color="#59A14F")
plt.title("Menciones específicas detectadas por regex")
plt.xlabel("% de iniciativas")
plt.ylabel("")
guardar_figura("09_menciones_regex.png")


**Interpretación.** Las expresiones regulares entregan indicadores transparentes de presencia temática. No sustituyen un análisis semántico profundo, pero son útiles para medir preguntas concretas de forma reproducible.


## 13. KWIC y WordTree simplificado

KWIC (*keyword in context*) muestra palabras alrededor de un término de interés. Esto permite revisar cómo se usa un concepto en textos reales. Además, se construye una visualización tipo WordTree simplificada como red de palabras frecuentes alrededor del término.


In [ ]:
def kwic(corpus, termino, ventana=6, max_rows=12):
    termino_norm = normalizar_texto(termino)
    rows = []
    for i, texto in enumerate(corpus):
        tokens = normalizar_texto(texto).split()
        termino_tokens = termino_norm.split()
        ancho = len(termino_tokens)
        for pos in range(0, len(tokens) - ancho + 1):
            if tokens[pos:pos + ancho] == termino_tokens:
                izquierda = " ".join(tokens[max(0, pos - ventana):pos])
                centro = " ".join(tokens[pos:pos + ancho])
                derecha = " ".join(tokens[pos + ancho:pos + ancho + ventana])
                rows.append({"doc_id": i, "izquierda": izquierda, "termino": centro, "derecha": derecha})
                if len(rows) >= max_rows:
                    return pd.DataFrame(rows)
    return pd.DataFrame(rows)

terminos_kwic = ["Estado", "derecho", "familia", "medio ambiente", "seguridad", "educación", "agua", "animales", "pensiones"]
kwic_tables = {}
for termino in terminos_kwic:
    tabla = kwic(iniciativas["texto_natural"], termino, ventana=6, max_rows=15)
    kwic_tables[termino] = tabla
    tabla.to_csv(TABLE_DIR / f"kwic_{normalizar_texto(termino).replace(' ', '_')}.csv", index=False)
    print(f"\nKWIC: {termino}")
    display(tabla.head(6))

def wordtree_simplificado(termino, ventana=4, top_n=18):
    termino_norm = normalizar_texto(termino)
    contexto = Counter()
    for texto in iniciativas["texto_natural"]:
        tokens = normalizar_texto(texto).split()
        term_tokens = termino_norm.split()
        width = len(term_tokens)
        for pos in range(0, len(tokens) - width + 1):
            if tokens[pos:pos+width] == term_tokens:
                around = tokens[max(0, pos-ventana):pos] + tokens[pos+width:pos+width+ventana]
                contexto.update([t for t in around if t not in stopwords_es and len(t) >= 3])
    G = nx.Graph()
    G.add_node(termino_norm)
    for palabra, freq in contexto.most_common(top_n):
        G.add_edge(termino_norm, palabra, weight=freq)
    return G

for termino in ["derecho", "medio ambiente", "seguridad"]:
    G = wordtree_simplificado(termino)
    if len(G.nodes) <= 1:
        continue
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=RANDOM_STATE)
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    nx.draw_networkx_nodes(G, pos, node_color="#A0CBE8", edgecolors="#1F4E79", node_size=900)
    nx.draw_networkx_edges(G, pos, width=[1 + 4*w/max(weights) for w in weights], alpha=0.55)
    nx.draw_networkx_labels(G, pos, font_size=9)
    plt.title(f"WordTree simplificado: {termino}")
    plt.axis("off")
    guardar_figura(f"10_wordtree_{normalizar_texto(termino).replace(' ', '_')}.png")


**Interpretación.** KWIC permite validar los hallazgos cuantitativos con ejemplos textuales. Si un término aparece con alta frecuencia, el contexto muestra si se usa como demanda, crítica, definición institucional o mención general.


## 14. WordClouds informativas

Las nubes de palabras se incluyen como visualización complementaria. Antes de cada nube se revisan las frecuencias asociadas, porque una WordCloud no reemplaza tablas ni análisis contextual.


In [ ]:
texto_wc = " ".join(" ".join(tokens) for tokens in iniciativas["tokens_limpios"])
wc = WordCloud(width=1200, height=650, background_color="white", colormap="viridis", random_state=RANDOM_STATE, collocations=False).generate(texto_wc)
plt.figure(figsize=(12, 6.5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud global informativa")
guardar_figura("11_wordcloud_global.png")

for comision, grupo in iniciativas.groupby("Comisión"):
    freqs = top_tokens(grupo["tokens_limpios"], 20)
    freqs.to_csv(TABLE_DIR / f"frecuencias_wordcloud_{normalizar_texto(comision).replace(' ', '_')[:40]}.csv", index=False)
    texto = " ".join(" ".join(tokens) for tokens in grupo["tokens_limpios"])
    if not texto.strip():
        continue
    wc = WordCloud(width=1000, height=550, background_color="white", colormap="magma", random_state=RANDOM_STATE, collocations=False).generate(texto)
    plt.figure(figsize=(10, 5.5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud por comisión: {comision}")
    guardar_figura(f"12_wordcloud_comision_{normalizar_texto(comision).replace(' ', '_')[:40]}.png")


**Interpretación.** Las WordClouds son útiles para comunicación visual, pero su lectura debe subordinarse a tablas de frecuencia, bigramas y ejemplos. Su valor principal es sintetizar visualmente patrones ya cuantificados.


## 15. Red de bigramas

La red de bigramas representa palabras como nodos y co-ocurrencias de dos palabras como aristas. Esta visualización permite observar comunidades léxicas o asociaciones recurrentes.


In [ ]:
def grafo_bigramas(df_bigramas, top_n=30):
    G = nx.Graph()
    for _, row in df_bigramas.head(top_n).iterrows():
        parts = row["ngram"].split()
        if len(parts) == 2:
            G.add_edge(parts[0], parts[1], weight=float(row["frecuencia"]))
    return G

G = grafo_bigramas(top_bigramas_global, top_n=30)
plt.figure(figsize=(11, 8))
pos = nx.spring_layout(G, seed=RANDOM_STATE, k=0.65)
weights = [G[u][v]["weight"] for u, v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_size=800, node_color="#A0CBE8", edgecolors="#1F4E79")
nx.draw_networkx_edges(G, pos, width=[1 + 4*w/max(weights) for w in weights], alpha=0.55)
nx.draw_networkx_labels(G, pos, font_size=9)
plt.title("Red global de bigramas frecuentes")
plt.axis("off")
guardar_figura("13_red_bigramas_global.png")


**Interpretación.** La red permite ver asociaciones léxicas recurrentes. Debe leerse como mapa exploratorio y no como estructura causal: dos palabras conectadas indican co-ocurrencia frecuente, no necesariamente una relación conceptual estable.


## 16. Comparación de vocabularios entre comisiones

Siguiendo la sugerencia del notebook de clase, se agregan diagramas de Venn entre comisiones. Como hay cuatro comisiones, se generan diagramas por tríos de comisiones: cada gráfico muestra palabras o bigramas representativos que son exclusivos de una comisión o compartidos entre dos o tres comisiones.

Se construyen dos familias de Venn:

- **Venn de palabras representativas:** usa los tokens más frecuentes por comisión.
- **Venn de bigramas representativos:** usa los bigramas más frecuentes por comisión.

Las áreas de los círculos no son proporcionales; la visualización busca mostrar contenido textual directamente dentro del diagrama. Las tablas exportadas permiten auditar las intersecciones completas.


In [ ]:
vocab_por_comision = {
    comision: set(tfidf_comision.loc[tfidf_comision["Comisión"].eq(comision), "termino"].head(30))
    for comision in tfidf_comision["Comisión"].unique()
}
inter_rows = []
for c1, c2 in itertools.combinations(vocab_por_comision, 2):
    inter = vocab_por_comision[c1] & vocab_por_comision[c2]
    union = vocab_por_comision[c1] | vocab_por_comision[c2]
    inter_rows.append({
        "comision_1": c1,
        "comision_2": c2,
        "interseccion": len(inter),
        "jaccard": round(len(inter) / len(union), 3) if union else 0,
        "terminos_compartidos": ", ".join(sorted(inter)[:15]),
    })
vocab_intersections = pd.DataFrame(inter_rows)
vocab_intersections.to_csv(TABLE_DIR / "interseccion_vocabularios_comisiones.csv", index=False)
display(vocab_intersections)

def abreviar_comision(nombre):
    nombre_limpio = normalizar_texto(nombre, quitar_numeros=True)
    if "economicos" in nombre_limpio:
        return "DESC-A"
    if "jurisdiccional" in nombre_limpio:
        return "FJ-OA"
    if "principios" in nombre_limpio:
        return "PDCP"
    if "sistema politico" in nombre_limpio:
        return "SP-RCFE"
    return nombre[:12]

def wrap_terms(items, max_items=5, max_width=28):
    selected = list(items)[:max_items]
    if not selected:
        return "sin términos"
    lines = []
    for item in selected:
        item = str(item)
        if len(item) > max_width:
            item = item[:max_width - 1] + "…"
        lines.append(item)
    return "\n".join(lines)

def venn_regions_3(A, B, C):
    return {
        "solo_A": sorted(A - B - C),
        "solo_B": sorted(B - A - C),
        "solo_C": sorted(C - A - B),
        "AB": sorted((A & B) - C),
        "AC": sorted((A & C) - B),
        "BC": sorted((B & C) - A),
        "ABC": sorted(A & B & C),
    }

def dibujar_venn_comisiones(trio, sets, tipo, filename):
    A, B, C = sets
    regs = venn_regions_3(A, B, C)
    labels = [abreviar_comision(c) for c in trio]

    fig, ax = plt.subplots(figsize=(11, 8))
    circles = [
        plt.Circle((-0.55, 0.0), 0.92, color="#4C78A8", alpha=0.25),
        plt.Circle((0.55, 0.0), 0.92, color="#F58518", alpha=0.25),
        plt.Circle((0.00, 0.78), 0.92, color="#54A24B", alpha=0.25),
    ]
    for circle in circles:
        ax.add_patch(circle)

    # Etiquetas de conjuntos
    ax.text(-1.20, -0.75, labels[0], ha="center", fontsize=11, fontweight="bold")
    ax.text(1.20, -0.75, labels[1], ha="center", fontsize=11, fontweight="bold")
    ax.text(0.00, 1.70, labels[2], ha="center", fontsize=11, fontweight="bold")

    # Regiones con palabras/bigramas, no solo conteos.
    ax.text(-0.92, -0.25, wrap_terms(regs["solo_A"], 5), ha="center", va="center", fontsize=8)
    ax.text(0.92, -0.25, wrap_terms(regs["solo_B"], 5), ha="center", va="center", fontsize=8)
    ax.text(0.00, 1.18, wrap_terms(regs["solo_C"], 5), ha="center", va="center", fontsize=8)
    ax.text(0.00, -0.42, wrap_terms(regs["AB"], 4), ha="center", va="center", fontsize=8)
    ax.text(-0.42, 0.48, wrap_terms(regs["AC"], 4), ha="center", va="center", fontsize=8)
    ax.text(0.42, 0.48, wrap_terms(regs["BC"], 4), ha="center", va="center", fontsize=8)
    ax.text(0.00, 0.22, wrap_terms(regs["ABC"], 4), ha="center", va="center", fontsize=8, fontweight="bold")

    ax.set_title(f"Venn de {tipo} representativos entre comisiones")
    ax.set_xlim(-1.75, 1.75)
    ax.set_ylim(-1.15, 1.95)
    ax.axis("off")
    guardar_figura(filename)
    return regs

venn_funcionales = {"debe", "deben", "ser", "sera", "sean", "puede", "pueden"}

def palabra_venn_ok(term):
    term = normalizar_texto(term, quitar_numeros=True)
    return len(term) >= 3 and term not in stopwords_nominales and term not in venn_funcionales

def bigrama_venn_ok(ngram):
    partes = normalizar_texto(ngram, quitar_numeros=True).split()
    if len(partes) != 2:
        return False
    if any(p in venn_funcionales for p in partes):
        return False
    return any(p not in stopwords_nominales for p in partes)

palabras_por_comision = {
    comision: set(
        top_tokens_comision.loc[top_tokens_comision["Comisión"].eq(comision)]
        .sort_values("frecuencia", ascending=False)
        ["termino"]
        .loc[lambda s: s.map(palabra_venn_ok)]
        .head(25)
    )
    for comision in top_tokens_comision["Comisión"].unique()
}
bigramas_por_comision_sets = {
    comision: set(
        bigramas_comision.loc[bigramas_comision["Comisión"].eq(comision)]
        .sort_values("frecuencia", ascending=False)
        ["ngram"]
        .loc[lambda s: s.map(bigrama_venn_ok)]
        .head(25)
    )
    for comision in bigramas_comision["Comisión"].unique()
}

venn_comision_rows = []
comisiones_ordenadas = tabla_comision["Comisión"].tolist()
for trio in itertools.combinations(comisiones_ordenadas, 3):
    slug = "_".join(abreviar_comision(c).lower().replace("-", "_") for c in trio)
    for tipo, mapping, prefix in [
        ("palabras", palabras_por_comision, "16a_venn_comisiones_palabras"),
        ("bigramas", bigramas_por_comision_sets, "16b_venn_comisiones_bigramas"),
    ]:
        sets = [mapping[c] for c in trio]
        regs = dibujar_venn_comisiones(
            trio,
            sets,
            tipo,
            f"{prefix}_{slug}.png",
        )
        for region, items in regs.items():
            venn_comision_rows.append({
                "tipo": tipo,
                "comision_A": trio[0],
                "comision_B": trio[1],
                "comision_C": trio[2],
                "region": region,
                "n_elementos": len(items),
                "elementos": ", ".join(items),
            })

venn_comisiones_df = pd.DataFrame(venn_comision_rows)
venn_comisiones_df.to_csv(TABLE_DIR / "venn_comisiones_palabras_bigramas_detalle.csv", index=False)
display(venn_comisiones_df.head(20))


**Interpretación.** Los Venn entre comisiones permiten distinguir tres situaciones: vocabulario exclusivo de una comisión, vocabulario compartido entre dos comisiones y vocabulario transversal. Cuando se usan bigramas, el diagrama es más informativo que con palabras aisladas, porque expresiones como `medio ambiente`, `poder judicial` o `partidos politicos` conservan más contexto.


## 17. Modelos clásicos de tópicos: NMF y LDA

Los modelos de tópicos se presentan como análisis complementario. Se entrenan NMF y LDA con varios valores de `k`, se revisan palabras principales y se selecciona una configuración interpretable. Estos modelos no reemplazan la lectura sustantiva ni el diccionario temático.


In [ ]:
def top_palabras_modelo(model, terms, n=10):
    rows = []
    for topic_idx, weights in enumerate(model.components_, start=1):
        order = weights.argsort()[::-1][:n]
        rows.append({
            "topico": topic_idx,
            "terminos": ", ".join(terms[order]),
        })
    return pd.DataFrame(rows)

topic_summaries = []
nmf_models = {}
for k in [5, 8, 10, 12]:
    model = NMF(n_components=k, random_state=RANDOM_STATE, init="nndsvda", max_iter=600)
    W = model.fit_transform(X_tfidf)
    terms_table = top_palabras_modelo(model, tfidf_terms, n=10)
    terms_table["modelo"] = "NMF"
    terms_table["k"] = k
    terms_table.to_csv(TABLE_DIR / f"nmf_topicos_k{k}.csv", index=False)
    nmf_models[k] = (model, W, terms_table)
    topic_summaries.append({"modelo": "NMF", "k": k, "nota": "Revisar interpretabilidad de términos principales"})

selected_k = 8
nmf_final, W_nmf, nmf_topics = nmf_models[selected_k]
iniciativas["topico_nmf"] = W_nmf.argmax(axis=1) + 1
iniciativas["peso_topico_nmf"] = W_nmf.max(axis=1)
nmf_topics.to_csv(TABLE_DIR / "topicos_nmf_final.csv", index=False)
display(nmf_topics)

count_vectorizer = CountVectorizer(
    preprocessor=lambda x: normalizar_texto(x, quitar_numeros=True),
    stop_words=list(stopwords_es),
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    max_features=3000,
)
X_count = count_vectorizer.fit_transform(iniciativas["texto_completo"])
count_terms = np.array(count_vectorizer.get_feature_names_out())
lda = LatentDirichletAllocation(n_components=8, random_state=RANDOM_STATE, learning_method="batch", max_iter=20)
lda_doc = lda.fit_transform(X_count)
iniciativas["topico_lda"] = lda_doc.argmax(axis=1) + 1
lda_topics = top_palabras_modelo(lda, count_terms, n=10)
lda_topics.to_csv(TABLE_DIR / "topicos_lda.csv", index=False)
display(lda_topics)

nmf_cross = pd.crosstab(iniciativas["Comisión"], iniciativas["topico_nmf"], normalize="index").round(3)
nmf_cross.to_csv(TABLE_DIR / "topicos_nmf_por_comision.csv")
plt.figure(figsize=(9, 4.8))
sns.heatmap(nmf_cross, annot=True, fmt=".0%", cmap="Blues")
plt.title("Tópicos NMF por comisión")
plt.xlabel("Tópico NMF")
plt.ylabel("Comisión")
guardar_figura("14_topicos_nmf_por_comision.png")


## 18. Diagramas de Venn entre palabras representativas de tópicos por comisión

Para complementar la lectura de tópicos, se comparan las palabras representativas de los tópicos NMF más presentes dentro de cada comisión. La lógica es la siguiente:

1. Para cada comisión se identifican los tres tópicos NMF dominantes por cantidad de iniciativas.
2. Para cada tópico se toma el conjunto de palabras representativas extraídas de sus términos principales.
3. Se dibuja un diagrama de Venn simplificado para observar qué palabras son exclusivas de cada tópico y cuáles se comparten.

Como no se asume instalada la librería `matplotlib-venn`, se construye una visualización propia con círculos superpuestos de Matplotlib. Las áreas no son proporcionales; el objetivo es interpretativo y se complementa con tablas exportadas de intersecciones.


In [ ]:
def palabras_de_topico(terminos, stopwords_extra=None):
    stop = set(stopwords_nominales) if "stopwords_nominales" in globals() else set(stopwords_es)
    if stopwords_extra:
        stop |= set(stopwords_extra)
    palabras = set()
    for termino in str(terminos).split(","):
        termino = normalizar_texto(termino.strip(), quitar_numeros=True)
        for palabra in termino.split():
            if len(palabra) >= 3 and palabra not in stop:
                palabras.add(palabra)
    return palabras

def slug_corto(texto, max_len=45):
    slug = normalizar_texto(texto, quitar_numeros=True).replace(" ", "_")
    slug = re.sub(r"_+", "_", slug).strip("_")
    return slug[:max_len]

topic_word_sets = {
    int(row["topico"]): palabras_de_topico(row["terminos"])
    for _, row in nmf_topics.iterrows()
}
topic_term_labels = {
    int(row["topico"]): row["terminos"]
    for _, row in nmf_topics.iterrows()
}

venn_rows = []
venn_detail_rows = []

def dibujar_venn_simplificado(comision, selected_topics, sets):
    labels = [f"T{t}" for t in selected_topics]
    A, B, C = sets
    AB = A & B
    AC = A & C
    BC = B & C
    ABC = A & B & C
    only_A = A - B - C
    only_B = B - A - C
    only_C = C - A - B

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    circles = [
        plt.Circle((-0.45, 0.05), 0.78, color="#4C78A8", alpha=0.28),
        plt.Circle((0.45, 0.05), 0.78, color="#F58518", alpha=0.28),
        plt.Circle((0.00, 0.68), 0.78, color="#54A24B", alpha=0.28),
    ]
    for circle in circles:
        ax.add_patch(circle)

    ax.text(-0.88, -0.58, f"{labels[0]}\nsolo: {len(only_A)}", ha="center", va="center", fontsize=10)
    ax.text(0.88, -0.58, f"{labels[1]}\nsolo: {len(only_B)}", ha="center", va="center", fontsize=10)
    ax.text(0.00, 1.25, f"{labels[2]}\nsolo: {len(only_C)}", ha="center", va="center", fontsize=10)
    ax.text(0.00, -0.13, f"{labels[0]}∩{labels[1]}\n{len(AB)}", ha="center", va="center", fontsize=9)
    ax.text(-0.34, 0.54, f"{labels[0]}∩{labels[2]}\n{len(AC)}", ha="center", va="center", fontsize=9)
    ax.text(0.34, 0.54, f"{labels[1]}∩{labels[2]}\n{len(BC)}", ha="center", va="center", fontsize=9)
    ax.text(0.00, 0.33, f"triple\n{len(ABC)}", ha="center", va="center", fontsize=10, fontweight="bold")

    comunes = ", ".join(sorted(ABC)) if ABC else "sin palabras comunes en los tres tópicos"
    ax.text(
        0, -1.08,
        f"Palabras comunes a los tres: {comunes}",
        ha="center",
        va="center",
        fontsize=9,
        wrap=True,
    )
    ax.set_title(f"Venn simplificado de palabras de tópicos NMF\n{comision}")
    ax.set_xlim(-1.65, 1.65)
    ax.set_ylim(-1.25, 1.65)
    ax.axis("off")
    guardar_figura(f"14b_venn_topicos_comision_{slug_corto(comision)}.png")

    return {
        "solo_1": sorted(only_A),
        "solo_2": sorted(only_B),
        "solo_3": sorted(only_C),
        "inter_12": sorted(AB),
        "inter_13": sorted(AC),
        "inter_23": sorted(BC),
        "inter_123": sorted(ABC),
    }

for comision, grupo in iniciativas.groupby("Comisión"):
    dominant_topics = (
        grupo["topico_nmf"]
        .value_counts()
        .head(3)
        .index
        .astype(int)
        .tolist()
    )
    if len(dominant_topics) < 3:
        continue
    sets = [topic_word_sets[t] for t in dominant_topics]
    intersections = dibujar_venn_simplificado(comision, dominant_topics, sets)

    counts = grupo["topico_nmf"].value_counts().to_dict()
    venn_rows.append({
        "Comisión": comision,
        "topico_1": dominant_topics[0],
        "topico_2": dominant_topics[1],
        "topico_3": dominant_topics[2],
        "n_topico_1_en_comision": int(counts.get(dominant_topics[0], 0)),
        "n_topico_2_en_comision": int(counts.get(dominant_topics[1], 0)),
        "n_topico_3_en_comision": int(counts.get(dominant_topics[2], 0)),
        "palabras_topico_1": ", ".join(sorted(sets[0])),
        "palabras_topico_2": ", ".join(sorted(sets[1])),
        "palabras_topico_3": ", ".join(sorted(sets[2])),
        "interseccion_triple": ", ".join(intersections["inter_123"]),
        "n_interseccion_triple": len(intersections["inter_123"]),
    })

    for tipo, palabras in intersections.items():
        venn_detail_rows.append({
            "Comisión": comision,
            "topicos_comparados": "-".join([f"T{t}" for t in dominant_topics]),
            "tipo_interseccion": tipo,
            "n_palabras": len(palabras),
            "palabras": ", ".join(palabras),
        })

venn_resumen = pd.DataFrame(venn_rows)
venn_detalle = pd.DataFrame(venn_detail_rows)
venn_resumen.to_csv(TABLE_DIR / "venn_topicos_por_comision_resumen.csv", index=False)
venn_detalle.to_csv(TABLE_DIR / "venn_topicos_por_comision_detalle.csv", index=False)
display(venn_resumen)
display(venn_detalle.head(20))


**Interpretación.** Los diagramas de Venn ayudan a ver si los tópicos dominantes dentro de una comisión comparten vocabulario o si separan campos léxicos distintos. Cuando hay mucha intersección, los tópicos pueden estar capturando variaciones de un mismo campo temático. Cuando hay poca intersección, el clustering temático es más diferenciado. Esta lectura debe complementarse con los documentos representativos de cada tópico.


**Interpretación.** NMF y LDA ayudan a agrupar patrones léxicos latentes. Sin embargo, en textos breves estos tópicos pueden mezclarse o capturar fórmulas repetidas. Por eso se interpretan como complemento de métodos más transparentes.


## 19. Embeddings semánticos y BERTopic como análisis avanzado complementario

Esta sección ejecuta el flujo avanzado de embeddings semánticos, reducción UMAP y BERTopic. Se usa un modelo multilingüe de `sentence-transformers` para representar cada iniciativa como vector semántico. Luego UMAP proyecta esos vectores a dos dimensiones y BERTopic agrupa documentos a partir de esos embeddings.

Esta parte sigue siendo complementaria: ayuda a observar patrones latentes, pero no reemplaza el análisis principal basado en frecuencia, bigramas, TF-IDF, diccionario propio, KWIC y visualizaciones interpretativas.


In [ ]:
embeddings_path = MODEL_DIR / "embeddings_sentence_transformer.npy"
embeddings_status = {"ejecutado": False, "motivo": ""}
embeddings = None

if RUN_EMBEDDINGS and HAS_EMBEDDINGS:
    try:
        if embeddings_path.exists():
            embeddings = np.load(embeddings_path)
            origen_embeddings = "cache local"
        else:
            model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
            st_model = SentenceTransformer(model_name)
            embeddings = st_model.encode(
                iniciativas["texto_natural"].tolist(),
                show_progress_bar=True,
                normalize_embeddings=True,
            )
            np.save(embeddings_path, embeddings)
            origen_embeddings = model_name

        reducer = umap.UMAP(n_neighbors=15, min_dist=0.05, random_state=RANDOM_STATE)
        emb_2d = reducer.fit_transform(embeddings)
        iniciativas["emb_x"] = emb_2d[:, 0]
        iniciativas["emb_y"] = emb_2d[:, 1]
        iniciativas[["Nº", "Título", "Comisión", "Capítulo", "emb_x", "emb_y"]].to_csv(
            TABLE_DIR / "embeddings_umap_documentos.csv",
            index=False,
        )

        plt.figure(figsize=(9, 6))
        sns.scatterplot(data=iniciativas, x="emb_x", y="emb_y", hue="Comisión", s=28, alpha=0.8)
        plt.title("Mapa 2D de embeddings por comisión")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        guardar_figura("15_mapa_embeddings_comision.png")

        plt.figure(figsize=(9, 6))
        sns.scatterplot(data=iniciativas, x="emb_x", y="emb_y", hue="Más de 10mil apoyos", palette="Set1", s=28, alpha=0.8)
        plt.title("Mapa 2D de embeddings según más de 10 mil apoyos")
        guardar_figura("16_mapa_embeddings_mas_10mil.png")

        embeddings_status = {
            "ejecutado": True,
            "motivo": f"Embeddings calculados desde {origen_embeddings}",
            "n_documentos": int(embeddings.shape[0]),
            "dimensiones": int(embeddings.shape[1]),
        }
    except Exception as exc:
        embeddings = None
        embeddings_status = {
            "ejecutado": False,
            "motivo": f"No se pudieron calcular embeddings: {type(exc).__name__}: {exc}",
        }
else:
    embeddings_status = {
        "ejecutado": False,
        "motivo": "RUN_EMBEDDINGS=False o dependencias no disponibles.",
    }
print(embeddings_status["motivo"])

bertopic_status = {"ejecutado": False, "motivo": ""}
if RUN_BERTOPIC and HAS_BERTOPIC and embeddings is not None:
    try:
        configs = [
            ("fino", 10, 0.0, 20, 5),
            ("balanceado", 15, 0.0, 30, 10),
            ("estable", 30, 0.0, 40, 10),
        ]
        bertopic_rows = []
        best = None
        best_score = np.inf

        for nombre, n_neighbors, min_dist, min_cluster_size, min_samples in configs:
            t0 = time.time()
            umap_model = UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=RANDOM_STATE)
            hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, prediction_data=True)
            vectorizer_model = CountVectorizer(
                ngram_range=(1, 2),
                stop_words=list(stopwords_es),
                min_df=2,
            )
            topic_model = BERTopic(
                umap_model=umap_model,
                hdbscan_model=hdbscan_model,
                vectorizer_model=vectorizer_model,
                language="multilingual",
                calculate_probabilities=False,
                verbose=False,
            )
            topics, _ = topic_model.fit_transform(iniciativas["texto_natural"].tolist(), embeddings)
            info = topic_model.get_topic_info()
            outlier_rate = float(np.mean(np.array(topics) == -1))
            n_topics = int((info["Topic"] != -1).sum())
            dominant_share = float(info.loc[info["Topic"] != -1, "Count"].max() / len(iniciativas)) if n_topics > 0 else 1.0
            score = outlier_rate + dominant_share + abs(n_topics - 12) / 100
            elapsed = round(time.time() - t0, 1)
            bertopic_rows.append({
                "configuracion": nombre,
                "n_neighbors": n_neighbors,
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "n_topicos_sin_outlier": n_topics,
                "tasa_outliers": round(outlier_rate, 3),
                "participacion_topico_dominante": round(dominant_share, 3),
                "tiempo_segundos": elapsed,
                "score_seleccion": round(score, 3),
            })
            if score < best_score:
                best_score = score
                best = {
                    "configuracion": nombre,
                    "model": topic_model,
                    "topics": topics,
                    "info": info,
                }

        bertopic_eval = pd.DataFrame(bertopic_rows).sort_values("score_seleccion")
        bertopic_eval.to_csv(TABLE_DIR / "bertopic_comparacion_configuraciones.csv", index=False)
        display(bertopic_eval)

        if best is not None:
            iniciativas["topico_bertopic"] = best["topics"]
            best["info"].to_csv(TABLE_DIR / "bertopic_topicos_final.csv", index=False)
            iniciativas[["Nº", "Título", "Comisión", "Capítulo", "Nº de apoyos", "topico_bertopic"]].to_csv(
                TABLE_DIR / "documentos_topicos_bertopic.csv",
                index=False,
            )
            bertopic_cross = pd.crosstab(
                iniciativas["Comisión"],
                iniciativas["topico_bertopic"],
                normalize="index",
            ).round(3)
            bertopic_cross.to_csv(TABLE_DIR / "bertopic_topicos_por_comision.csv")

            plt.figure(figsize=(12, 4.8))
            plot_info = best["info"][best["info"]["Topic"] != -1].head(15)
            sns.barplot(data=plot_info, y="Name", x="Count", color="#4C78A8")
            plt.title(f"BERTopic: tópicos principales ({best['configuracion']})")
            plt.xlabel("Documentos")
            plt.ylabel("Tópico")
            guardar_figura("17_bertopic_topicos_principales.png")

            if "emb_x" in iniciativas.columns and "emb_y" in iniciativas.columns:
                plot_df = iniciativas.copy()
                plot_df["topico_bertopic_plot"] = plot_df["topico_bertopic"].astype(str)
                plt.figure(figsize=(9, 6))
                sns.scatterplot(data=plot_df, x="emb_x", y="emb_y", hue="topico_bertopic_plot", s=26, alpha=0.78, legend=False)
                plt.title("Mapa 2D de embeddings coloreado por tópico BERTopic")
                guardar_figura("18_mapa_embeddings_bertopic.png")

            bertopic_status = {
                "ejecutado": True,
                "motivo": f"BERTopic ejecutado; configuración seleccionada: {best['configuracion']}",
                "configuracion_final": best["configuracion"],
            }
    except Exception as exc:
        bertopic_status = {
            "ejecutado": False,
            "motivo": f"No se pudo ejecutar BERTopic: {type(exc).__name__}: {exc}",
        }
else:
    bertopic_status = {
        "ejecutado": False,
        "motivo": "RUN_BERTOPIC=False, sin embeddings o dependencias no disponibles.",
    }
print(bertopic_status["motivo"])

pd.DataFrame([embeddings_status]).to_csv(TABLE_DIR / "estado_embeddings.csv", index=False)
pd.DataFrame([bertopic_status]).to_csv(TABLE_DIR / "estado_bertopic.csv", index=False)


**Interpretación.** Los embeddings y BERTopic pueden aportar una lectura semántica complementaria, pero no deben desplazar el análisis principal. En esta entrega se prioriza que el flujo central sea reproducible, explicable y auditable.


## 20. Comparación integrada de métodos

Esta tabla resume qué aporta cada técnica, cuál es su principal salida y qué limitación debe considerarse al interpretarla.


In [ ]:
comparacion_metodos = pd.DataFrame([
    ("Frecuencia", "Identifica términos dominantes", "Top tokens globales y por grupo", "No entrega contexto por sí sola"),
    ("Bigramas", "Agrega contexto léxico", "Expresiones frecuentes de dos palabras", "Sigue siendo conteo superficial"),
    ("Nominales candidatos", "Identifica expresiones nominales informativas", "Sintagmas y nombres compuestos frecuentes", "Es una aproximación estadística, no análisis sintáctico perfecto"),
    ("Componentes gramaticales", "Organiza piezas funcionales del discurso", "Red de nominales, acciones, instituciones, cifras y atributos", "Es heurístico y no reemplaza parsing gramatical formal"),
    ("TF-IDF", "Detecta términos distintivos", "Términos característicos por comisión", "Depende de parámetros min_df/max_df"),
    ("Clustering por comisión", "Detecta subgrupos temáticos internos", "Clusters KMeans dentro de cada comisión", "Los clusters requieren validación interpretativa"),
    ("Diccionario", "Entrega análisis supervisado e interpretable", "Prevalencia por categoría temática", "Depende de decisiones del investigador"),
    ("Regex/KWIC", "Valida menciones y contexto", "Tablas de presencia y concordancias", "No resuelve ambigüedad semántica"),
    ("WordCloud", "Comunica patrones frecuentes", "Nubes globales y por comisión", "Puede sobre simplificar"),
    ("Red de bigramas", "Visualiza asociaciones léxicas", "Grafo de co-ocurrencias", "No implica relación causal"),
    ("Venn entre comisiones", "Compara vocabulario representativo por comisión", "Venn de palabras y bigramas entre tríos de comisiones", "Las áreas no son proporcionales; con cuatro comisiones se usan tríos"),
    ("Venn de tópicos", "Compara vocabularios representativos entre tópicos", "Diagramas por comisión e intersecciones léxicas", "Las áreas no son proporcionales; requiere lectura cualitativa"),
    ("NMF", "Agrupa patrones latentes", "Tópicos interpretables", "Puede mezclar temas en textos breves"),
    ("LDA", "Modelo probabilístico de tópicos", "Tópicos por distribución de palabras", "Sensible a textos cortos y parámetros"),
    ("Embeddings", "Representa similitud semántica", "Mapa 2D opcional", "Requiere modelos externos"),
    ("BERTopic", "Tópicos semánticos avanzados", "Tópicos con embeddings y clustering", "Costoso y sensible a configuración"),
], columns=["Método", "Qué aporta", "Principal resultado", "Limitación"])
comparacion_metodos.to_csv(TABLE_DIR / "comparacion_integrada_metodos.csv", index=False)
display(comparacion_metodos)


## 21. Hallazgos principales, limitaciones y conclusiones

A partir de los resultados anteriores, se sintetizan hallazgos académicos. Cada afirmación debe leerse junto con las tablas y visualizaciones exportadas; no se incorporan resultados inventados ni interpretaciones no respaldadas por evidencia del procesamiento.


In [ ]:
hallazgos = [
    "La base posee suficiente completitud textual para análisis PLN, con muy pocos textos vacíos en las columnas centrales.",
    "La distribución institucional de iniciativas está concentrada en algunas comisiones y capítulos, por lo que el análisis textual debe considerar esa estructura previa.",
    "Los conteos de tokens y bigramas muestran ejes recurrentes relacionados con derechos, vida, educación, salud, medio ambiente, sistema político y justicia.",
    "La extracción de nominales candidatos permite identificar expresiones compuestas más informativas que tokens aislados.",
    "Los diagramas de Venn entre comisiones permiten observar vocabularios exclusivos y compartidos usando palabras y bigramas representativos.",
    "TF-IDF permite distinguir vocabularios característicos por comisión, complementando la frecuencia global.",
    "El clustering separado por comisión permite explorar subtemas internos sin confundirlos con la clasificación institucional general.",
    "El diccionario temático constitucional propio aporta trazabilidad, auditabilidad e interpretación sustantiva de las menciones.",
    "Las iniciativas con más apoyos no deben interpretarse como opinión general de la población, sino como señales de adhesión dentro de un mecanismo participativo específico.",
    "Los modelos NMF y LDA ofrecen agrupaciones complementarias, pero requieren lectura crítica por la brevedad de los textos.",
]
limitaciones = [
    "Los textos son breves y pueden no contener todo el argumento de cada iniciativa.",
    "El apoyo ciudadano mide adhesión dentro del proceso participativo, no opinión pública general.",
    "La clasificación por comisión y capítulo introduce una estructura institucional previa.",
    "Los modelos de tópicos no deben interpretarse sin revisión sustantiva de términos y ejemplos.",
    "El diccionario temático depende de decisiones del investigador y debe mejorarse iterativamente.",
]
conclusiones = [
    "El PLN permite transformar un conjunto amplio de iniciativas ciudadanas en evidencia temática ordenada.",
    "La combinación de métodos exploratorios, supervisados y no supervisados permite equilibrar amplitud descriptiva e interpretabilidad.",
    "El diccionario propio es el aporte metodológico más trazable, porque explicita categorías, términos y criterios de inclusión.",
    "Los modelos de tópicos y embeddings pueden aportar patrones latentes, pero deben mantenerse como complemento crítico.",
]
sintesis = pd.DataFrame({
    "tipo": ["hallazgo"] * len(hallazgos) + ["limitacion"] * len(limitaciones) + ["conclusion"] * len(conclusiones),
    "texto": hallazgos + limitaciones + conclusiones,
})
sintesis.to_csv(TABLE_DIR / "sintesis_hallazgos_limitaciones_conclusiones.csv", index=False)

print("Hallazgos principales")
for h in hallazgos:
    print("-", h)
print("\nLimitaciones")
for l in limitaciones:
    print("-", l)
print("\nConclusiones")
for c in conclusiones:
    print("-", c)


## 22. Exportación final

Se guarda la base limpia, tablas, figuras y archivos metodológicos. Estos archivos permiten auditar el análisis y reutilizar resultados en el informe Word.


In [ ]:
base_export = iniciativas.drop(columns=["tokens_limpios"], errors="ignore")
base_export.to_csv(OUTPUT_DIR / "base_limpia_ipn.csv", index=False)

resumen_export = {
    "n_iniciativas": int(len(iniciativas)),
    "n_comisiones": int(iniciativas["Comisión"].nunique()),
    "n_capitulos": int(iniciativas["Capítulo"].nunique()),
    "porcentaje_con_organizacion": round(100 * iniciativas["Organización"].notna().mean(), 2),
    "mediana_apoyos": float(iniciativas["Nº de apoyos"].median()),
    "max_apoyos": int(iniciativas["Nº de apoyos"].max()),
    "figuras": sorted(str(p) for p in FIG_DIR.glob("*.png")),
    "tablas": sorted(str(p) for p in TABLE_DIR.glob("*.csv")),
    "diccionario": sorted(str(p) for p in DICT_DIR.glob("*")),
}
(OUTPUT_DIR / "resumen_ejecucion.json").write_text(json.dumps(resumen_export, ensure_ascii=False, indent=2), encoding="utf-8")
print("Base limpia:", OUTPUT_DIR / "base_limpia_ipn.csv")
print("Resumen:", OUTPUT_DIR / "resumen_ejecucion.json")
print("Figuras:", len(resumen_export["figuras"]))
print("Tablas:", len(resumen_export["tablas"]))


## 23. Síntesis metodológica y aprendizajes

Este trabajo muestra cómo el Procesamiento de Lenguaje Natural permite transformar un conjunto amplio de iniciativas ciudadanas en evidencia temática ordenada. El análisis combina métodos exploratorios, supervisados y no supervisados, manteniendo como eje las técnicas más interpretables.

El diccionario temático constitucional propio entrega trazabilidad e interpretabilidad, porque explicita las categorías, términos y criterios de inclusión. Los modelos de tópicos y embeddings aportan una lectura complementaria de patrones latentes, pero no deben sobreinterpretarse. Todo resultado sustantivo debe estar respaldado por tablas, gráficos o ejemplos textuales de la base.

### Checklist final

- Notebook ejecutado completo.
- Datos cargados correctamente.
- Tablas exportadas.
- Figuras exportadas.
- Modelos clásicos ejecutados.
- Nominales candidatos y clustering por comisión incorporados.
- Informe Word generado externamente a partir de los resultados exportados.
- Conclusiones redactadas.
- Limitaciones incluidas.
